In [1]:
print(123)

123


In [2]:
from openai import OpenAI
from dotenv import load_dotenv
load_dotenv()
import os
openai_client = OpenAI(
    api_key=os.getenv("GEMINI_API_KEY"),
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)

In [4]:
from rag_helper import RAGBase
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

In [6]:
instructions = """
You're a course teaching assistant.
Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.
""".strip()

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
    instructions=instructions,
)

In [8]:
assistant.rag("How do I run Ollama locally?")

'To run Ollama locally, follow these steps:\n\n1.  **Install Ollama:**\n    *   Visit [https://ollama.com/download](https://ollama.com/download) and select your operating system:\n        *   **macOS**: Download the `.pkg` and install it.\n        *   **Windows**: Download the `.msi` and install it.\n        *   **Linux**: Open your terminal and run:\n            ```bash\n            curl -fsSL https://ollama.com/install.sh | sh\n            ```\n\n2.  **Run a Model:**\n    *   Once installed, open a terminal and type:\n        ```bash\n        ollama run llama3\n        ```\n    *   This command will download the LLaMA 3 model (~4GB), start the model locally, and open a chat-like interface for you to type questions.\n\n3.  **Test the Local Server (Optional):**\n    *   To verify the Ollama local server is running, execute the following command in your terminal:\n        ```bash\n        curl http://localhost:11434\n        ```\n    *   You should receive a response similar to: `{"mode

In [10]:
assistant.rag("How do I run Olama locally?")

'I am sorry, but the provided context does not contain information on how to run Olama locally.'

In [80]:
messages = [
    {"role": "user", "content": "I just discovered the course. Can I join it?"}
]

response = openai_client.chat.completions.create(
    model="gemini-2.5-flash",
    messages=messages,
)

response.choices[0].message

ChatCompletionMessage(content='That\'s exciting! Discovering a new course is a great feeling.\n\nTo give you the most accurate answer, I need a little more information about the course you\'re referring to.\n\n**Could you tell me:**\n\n1.  **What is the name of the course or where did you find it?** (e.g., "Python for Beginners on Coursera," "Spring Semester Calculus at XYZ University," "The \'Intro to AI\' bootcamp," etc.)\n2.  **Is it an online self-paced course, a university program, a live workshop, a MOOC (Massive Open Online Course), etc.?**\n3.  **Does it have a specific start date, or is it continuous enrollment?**\n\n**Generally, here\'s what to look for:**\n\n*   **Official Website/Platform:** The best place to find enrollment information is always on the course\'s official website or the platform hosting it (like Coursera, edX, Udemy, a university\'s site, etc.).\n*   **Look for sections like:** "Enroll Now," "Admissions," "How to Apply," "Start Dates," "FAQ," or "Course Sch

In [12]:
def search(query):
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

In [61]:
search_tool = {
    "type": "function",
    "function": {
        "name": "search",
        "description": "Search the DataTalks.Club course FAQ database for information.",
        "parameters": {
            "type": "object",
            "properties": {
                "query": {
                    "type": "string"
                }
            },
            "required": ["query"]
        }
    }
}

In [67]:
response = openai_client.chat.completions.create(
    model="gemini-2.5-flash",
    messages=messages,
    tools=[search_tool]
)

In [57]:
call = response.choices[0].message

In [58]:
import json
tool_call = call.tool_calls[0]
args = json.loads(tool_call.function.arguments)
args

{'query': 'join course'}

In [43]:
results=search(**args)

In [45]:
result_json = json.dumps(results, indent=2)

In [46]:
print(result_json) 

[
  {
    "id": "74eb249bbf",
    "course": "llm-zoomcamp",
    "section": "General Course-Related Questions",
    "question": "I just discovered the course. Can I still join?",
    "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\u2019re still accepting submissions."
  },
  {
    "id": "bd31146b0e",
    "course": "llm-zoomcamp",
    "section": "General Course-Related Questions",
    "question": "When will the course be offered next?",
    "answer": "Summer 2027."
  },
  {
    "id": "04919992b3",
    "course": "llm-zoomcamp",
    "section": "General Course-Related Questions",
    "question": "How should I start the course and follow the weekly workflow?",
    "answer": "Start with the [LLM Zoomcamp docs](https://datatalks.club/docs/courses/llm-zoomcamp/), the [general Zoomcamp logistics docs](https://datatalks.club/docs/courses/zoomcamp-logistics/), and the [LLM Zoomcamp GitHub repository](https://github.com/DataTalksClub/llm-zoomcamp).

In [50]:
function_call_output={
    "type": "function_call_output",
    "call_id": tool_call.id,
    "Output": result_json
}

In [81]:
messages.append({
    "role": "tool",
    "name":"search",
    "tool_call_id": "function-call-12276860441250591489",
    "content": '{"id": "74eb249bbf", "course": "llm-zoomcamp", "section": "General course questions", "text": "Yes, you can still join the course..."}'
})

In [82]:
messages

[{'role': 'user', 'content': 'I just discovered the course. Can I join it?'},
 {'role': 'tool',
  'name': 'search',
  'tool_call_id': 'function-call-12276860441250591489',
  'content': '{"id": "74eb249bbf", "course": "llm-zoomcamp", "section": "General course questions", "text": "Yes, you can still join the course..."}'}]

In [84]:
from openai.types.chat import (
    ChatCompletionUserMessageParam,
    ChatCompletionAssistantMessageParam,
    ChatCompletionToolMessageParam
)

messages = [
    ChatCompletionUserMessageParam(
        role="user", 
        content="I just discovered the course. Can I join now?"
    ),
    ChatCompletionAssistantMessageParam(
        role="assistant",
        tool_calls=[
            {
                "id": "function-call-12276860441250591489",
                "type": "function",
                "function": {
                    "name": "search",
                    "arguments": '{"query":"join course discovered late can I join enroll late"}'
                }
            }
        ]
    ),
    ChatCompletionToolMessageParam(
        role="tool",
        name="search",  
        tool_call_id="function-call-12276860441250591489",
        content='{"id": "74eb249bbf", "course": "llm-zoomcamp", "section": "General course questions", "text": "Yes, you can still join the course. If you want to receive a certificate, you need to submit your project..."}'
    )
]

# 2. Fire the request with your clean payload
response = openai_client.chat.completions.create(
    model="gemini-2.5-flash",
    messages=messages,
    tools=[search_tool]
)

print(response.choices[0].message.content)

Yes, you can still join the course. If you want to receive a certificate, you will need to submit your project.


In [85]:
print(response.choices[0].message)

ChatCompletionMessage(content='Yes, you can still join the course. If you want to receive a certificate, you will need to submit your project.', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=None)


In [88]:

usage = response.usage
usage.prompt_tokens, usage.completion_tokens

(149, 25)

In [89]:
def calculate_gpt54mini_price(input_tokens, output_tokens):
    INPUT_PRICE_PER_MILLION = 0.15
    OUTPUT_PRICE_PER_MILLION = 0.60

    input_cost = (input_tokens / 1_000_000) * INPUT_PRICE_PER_MILLION
    output_cost = (output_tokens / 1_000_000) * OUTPUT_PRICE_PER_MILLION
    total_cost = input_cost + output_cost

    return {
        "input_cost": input_cost,
        "output_cost": output_cost,
        "total_cost": total_cost,
    }

result = calculate_gpt54mini_price(149, 25)
print("Total cost: $", round(result["total_cost"], 8))

Total cost: $ 3.735e-05
